# 📱 WhatsApp Outreach Tool — L&D Designs
**Turns your leads spreadsheet into a click-to-send WhatsApp dashboard.**

1. Run Cell 1 — upload your `wigan_hair_leads_*.xlsx` file
2. Run Cell 2 — generates your outreach dashboard
3. Run Cell 3 — downloads the dashboard as an HTML file
4. Open the HTML file in your browser and start messaging

In [ ]:
# ── Cell 1: Upload your leads spreadsheet ────────────────────────────────
from google.colab import files
import openpyxl

print('Select your wigan_hair_leads_*.xlsx file...')
uploaded = files.upload()
filename = list(uploaded.keys())[0]

wb = openpyxl.load_workbook(filename)
ws = wb['Leads']

headers = [cell.value for cell in ws[1]]
leads = []
for row in ws.iter_rows(min_row=2, values_only=True):
    if row[0]:
        leads.append(dict(zip(headers, row)))

leads_with_phone = [l for l in leads if l.get('Phone Number')]
print('\n✅ Loaded', len(leads), 'leads total')
print('  ', len(leads_with_phone), 'have a phone number (these get WhatsApp buttons)')
print('  ', len(leads) - len(leads_with_phone), 'have no phone number (shown as info only)')

In [ ]:
# ── Cell 2: Generate the outreach dashboard ───────────────────────────────
import re
from urllib.parse import quote
from datetime import datetime

# ── YOUR MESSAGE TEMPLATES ────────────────────────────────────────────────
NO_SITE_MSG = """Hi! I came across {name} and noticed you don't have a website yet.

I'm Dylan from L&D Designs - I build professional websites for local barbers and hairdressers across the Wigan area.

A website means new customers can find you on Google, see your work, and get in touch easily. Most of my sites are live within 1-2 weeks.

Would you be up for a free quote? No pressure at all

- Dylan, L&D Designs"""

OUTDATED_MSG = """Hi! I came across {name} and noticed your website could do with a refresh.

I'm Dylan from L&D Designs - I build modern websites for local barbers and hairdressers across the Wigan area.

An updated site helps you rank higher on Google and gives customers a much better first impression.

Happy to offer you a free quote with no obligation at all

- Dylan, L&D Designs"""
# ─────────────────────────────────────────────────────────────────────────

def clean_phone(phone):
    if not phone:
        return ''
    digits = re.sub(r'[^\d]', '', str(phone))
    if digits.startswith('0'):
        digits = '44' + digits[1:]
    elif not digits.startswith('44'):
        digits = '44' + digits
    return digits

def make_wa_link(phone, message):
    clean = clean_phone(phone)
    if not clean:
        return ''
    return 'https://wa.me/' + clean + '?text=' + quote(message)

STATUS_INFO = {
    'NONE':        ('#ffd6d6', '#c0392b', 'No website'),
    'SOCIAL ONLY': ('#d6eaff', '#2980b9', 'Social only'),
    'OUTDATED':    ('#fff2cc', '#e67e22', 'Outdated site'),
    'ERROR':       ('#e8e8e8', '#7f8c8d', 'Site error'),
}

def make_card(lead):
    name    = str(lead.get('Business Name') or 'Unknown')
    phone   = str(lead.get('Phone Number') or '')
    email   = str(lead.get('Email Address') or '')
    status  = str(lead.get('Website Status') or '').upper().strip()
    address = str(lead.get('Address') or '')
    dist    = str(lead.get('Distance (miles)') or '')
    website = str(lead.get('Website / Social URL') or '')
    notes   = str(lead.get('Notes') or '')

    bg, accent, badge = STATUS_INFO.get(status, ('#f7f7f7', '#555', status))

    if 'OUTDATED' in status or 'SOCIAL' in status:
        message = OUTDATED_MSG.format(name=name)
    else:
        message = NO_SITE_MSG.format(name=name)

    wa_link = make_wa_link(phone, message)

    # Build optional lines separately to avoid quote conflicts in f-strings
    if wa_link:
        wa_button = '<a class="wa-btn" href="' + wa_link + '" target="_blank" onclick="markSent(this)">Send WhatsApp</a>'
    else:
        wa_button = '<span class="no-phone">No phone number</span>'

    email_line   = ('<div class="detail">&#9993; ' + email + '</div>') if email else ''
    site_display = (website[:50] + '...') if len(website) > 50 else website
    website_line = ('<div class="detail"><a href="' + website + '" target="_blank">' + site_display + '</a></div>') if website else ''
    notes_line   = ('<div class="detail muted">' + notes + '</div>') if notes else ''
    dist_display = (dist + ' mi') if dist else ''

    return (
        '<div class="card" style="border-left:4px solid ' + accent + '; background:' + bg + '">'
        '<div class="card-top">'
        '<div><div class="biz-name">' + name + '</div>'
        '<div class="badge" style="color:' + accent + '">' + badge + '</div></div>'
        '<div class="dist">' + dist_display + '</div></div>'
        '<div class="detail">&#128205; ' + address + '</div>'
        '<div class="detail">&#128222; ' + (phone if phone else '&mdash;') + '</div>'
        + email_line + website_line + notes_line +
        '<div class="actions">' + wa_button + '</div></div>'
    )

# Count by status
counts = {}
for l in leads:
    s = str(l.get('Website Status') or '').upper().strip()
    counts[s] = counts.get(s, 0) + 1

cards_html = '\n'.join(make_card(l) for l in leads)
total = len(leads)
with_phone = len([l for l in leads if l.get('Phone Number')])
sent_target = str(with_phone)

html = '''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>L&D Designs Outreach Dashboard</title>
<style>
* { box-sizing: border-box; margin: 0; padding: 0; }
body { font-family: -apple-system, BlinkMacSystemFont, Segoe UI, sans-serif; background: #f0f2f5; }
.header { background: #1a2035; color: white; padding: 18px 24px; }
.header h1 { font-size: 1.3rem; }
.header p  { opacity: .65; font-size: .85rem; margin-top: 3px; }
.stats { display: flex; gap: 10px; flex-wrap: wrap; padding: 14px 20px;
          background: white; border-bottom: 1px solid #ddd; }
.stat { background: #f7f7f7; border-radius: 8px; padding: 10px 16px; text-align: center; min-width: 80px; }
.stat .n { font-size: 1.5rem; font-weight: 700; color: #1a2035; }
.stat .l { font-size: .72rem; color: #999; }
.bar { padding: 10px 20px; background: white; border-bottom: 1px solid #eee;
        display: flex; gap: 8px; flex-wrap: wrap; align-items: center; }
.fbtn { padding: 6px 14px; border: 2px solid #ddd; border-radius: 20px;
         background: white; cursor: pointer; font-size: .8rem; }
.fbtn.on { border-color: #1a2035; background: #1a2035; color: white; }
.progress { margin-left: auto; font-size: .82rem; color: #666; }
.grid { display: grid; grid-template-columns: repeat(auto-fill, minmax(320px, 1fr));
         gap: 12px; padding: 16px 20px; }
.card { background: white; border-radius: 8px; padding: 14px;
         box-shadow: 0 1px 3px rgba(0,0,0,.07); transition: opacity .3s; }
.card.sent { opacity: .35; }
.card-top { display: flex; justify-content: space-between; margin-bottom: 8px; }
.biz-name { font-weight: 700; font-size: .95rem; }
.badge { font-size: .72rem; font-weight: 600; margin-top: 2px; }
.dist { font-size: .78rem; color: #aaa; white-space: nowrap; padding-left: 6px; }
.detail { font-size: .8rem; color: #555; margin: 2px 0; word-break: break-word; }
.detail a { color: #2980b9; text-decoration: none; }
.muted { color: #bbb; font-style: italic; }
.actions { margin-top: 10px; display: flex; gap: 8px; align-items: center; }
.wa-btn { display: inline-block; background: #25D366; color: white;
            padding: 7px 16px; border-radius: 5px; text-decoration: none;
            font-size: .82rem; font-weight: 600; }
.wa-btn:hover { background: #1ebe5d; }
.undo-btn { background: none; border: 1px solid #ccc; color: #888;
              padding: 6px 10px; border-radius: 5px; cursor: pointer; font-size: .78rem; }
.no-phone { font-size: .78rem; color: #ccc; }
</style>
</head><body>
'''

html += '<div class="header"><h1>&#128248; L&amp;D Designs &mdash; WhatsApp Outreach</h1>'
html += '<p>50-mile radius of Wigan &nbsp;&middot;&nbsp; Generated ' + datetime.now().strftime('%d/%m/%Y %H:%M') + '</p></div>'

html += '<div class="stats">'
html += '<div class="stat"><div class="n">' + str(total) + '</div><div class="l">Total</div></div>'
html += '<div class="stat"><div class="n">' + str(with_phone) + '</div><div class="l">Have Phone</div></div>'
html += '<div class="stat"><div class="n" style="color:#c0392b">' + str(counts.get('NONE', 0)) + '</div><div class="l">No Website</div></div>'
html += '<div class="stat"><div class="n" style="color:#2980b9">' + str(counts.get('SOCIAL ONLY', 0)) + '</div><div class="l">Social Only</div></div>'
html += '<div class="stat"><div class="n" style="color:#e67e22">' + str(counts.get('OUTDATED', 0)) + '</div><div class="l">Outdated</div></div>'
html += '</div>'

html += '<div class="bar">'
html += '<button class="fbtn on" onclick="filt(this,\"all\")">All</button>'
html += '<button class="fbtn" onclick="filt(this,\"none\")">No Website</button>'
html += '<button class="fbtn" onclick="filt(this,\"social\")">Social Only</button>'
html += '<button class="fbtn" onclick="filt(this,\"outdated\")">Outdated</button>'
html += '<button class="fbtn" onclick="filt(this,\"unsent\")">Not Sent Yet</button>'
html += '<span class="progress">Sent: <strong id="sc">0</strong> / ' + sent_target + '</span>'
html += '</div>'

html += '<div class="grid" id="grid">' + cards_html + '</div>'

html += '''
<script>
function markSent(btn) {
  var card = btn.closest('.card');
  setTimeout(function() {
    card.classList.add('sent');
    if (!card.querySelector('.undo-btn')) {
      var u = document.createElement('button');
      u.className = 'undo-btn'; u.textContent = 'Undo';
      u.onclick = function() { card.classList.remove('sent'); u.remove(); updateCount(); };
      btn.parentNode.appendChild(u);
    }
    updateCount();
  }, 1500);
}
function updateCount() {
  document.getElementById('sc').textContent = document.querySelectorAll('.card.sent').length;
}
function filt(btn, type) {
  document.querySelectorAll('.fbtn').forEach(function(b) { b.classList.remove('on'); });
  btn.classList.add('on');
  document.querySelectorAll('.card').forEach(function(card) {
    var badge = (card.querySelector('.badge') ? card.querySelector('.badge').textContent : '').toLowerCase();
    var sent  = card.classList.contains('sent');
    var show  = true;
    if (type === 'none')     show = badge.indexOf('no website') > -1;
    if (type === 'social')   show = badge.indexOf('social') > -1;
    if (type === 'outdated') show = badge.indexOf('outdated') > -1;
    if (type === 'unsent')   show = !sent && card.querySelector('.wa-btn');
    card.style.display = show ? '' : 'none';
  });
}
</script></body></html>'''

with open('outreach_dashboard.html', 'w', encoding='utf-8') as f:
    f.write(html)

print('✅ Dashboard ready —', with_phone, 'WhatsApp buttons created')

In [ ]:
# ── Cell 3: Download the dashboard ───────────────────────────────────────
from google.colab import files
files.download('outreach_dashboard.html')
print('✅ Downloading...')
print()
print('Open outreach_dashboard.html in your browser.')
print('Click Send WhatsApp on each business — message is pre-typed, just hit Send.')
print('Cards fade out as you go so you can track who you have messaged.')